In [ ]:
from google.colab import files

In [ ]:
uploaded = files.upload()

In [ ]:
!pip install scipy
# # !pip install ultralytics
# # !pip install torch
# !pip install -r requirements.txt

In [ ]:
uploaded = files.upload()

In [ ]:
import numpy as np
import cv2
from moviepy.editor import VideoFileClip
from tensorflow import keras
model =keras.models.load_model('model.h5')
# model.summary()
# model1 = YOLO("yolov8n.pt")
# print('yolov8n',model1)
# model = torch.hub.load('ultralytics/yolov5','custom', 'yolov5/yolov5s.pt')
# print('yolov5s',model1.info)
# Generate the summary
# summary(model3, input_size=input_size)


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
def detect_lanes(image):
    # Preprocess the image
    image = cv2.resize(image, (160, 80))  # Resize to match input size
    image = image / 255.0  # Normalize pixel values

    # Perform prediction
    prediction = model.predict(np.expand_dims(image, axis=0))

    # Process the output
    lane_mask = prediction.squeeze()  # Remove batch dimension
    lane_mask = (lane_mask > 0.5).astype(np.uint8)  # Thresholding

    return lane_mask

def extract_lane_boundaries(lane_mask, original_image):
    # Step 1: Edge detection using Canny
    edges = cv2.Canny(lane_mask, 50, 150)

    # Step 2: Contour detection
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Step 3: Extract lane boundaries
    lane_boundaries = []
    for contour in contours:
        # Approximate the contour to reduce the number of points
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)

        # Skip small contours (noise)
        if cv2.contourArea(approx) < 100:
            continue

        # Add the approximated contour points to the list of lane boundaries
        lane_boundaries.append(approx)

    # Draw the detected lane boundaries on the original image
    result_image = original_image.copy()
    cv2.drawContours(result_image, lane_boundaries, -1, (0, 255, 0), 2)

    return result_image

# Main function
def main():
    # Path to the input video
    video_path = 'DetectingLanes.mp4'

    # Open the video file
    cap = cv2.VideoCapture(video_path)

    # Check if the video file opened successfully
    if not cap.isOpened():
        print("Error: Could not open video file.")
        return

    # Get the frames per second (fps) of the video
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Get the width and height of the video frames
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Create a VideoWriter object to save the processed video
    out = cv2.VideoWriter('output_video.avi', cv2.VideoWriter_fourcc(*'DIVX'), fps, (width, height))

    # Process each frame of the video
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Detect lanes in the frame
        lane_mask = detect_lanes(frame)

        # Extract lane boundaries
        result_frame = extract_lane_boundaries(lane_mask, frame)

        # Write the processed frame to the output video
        out.write(result_frame)

        # Check for 'q' key press to exit the loop
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the video capture and video writer objects
    cap.release()
    out.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


In [ ]:
class lanes():
  def __init__(self):
    self.recent_fit =[]
    self.avg_fit =[]

def road_lines(image):
    # Resize the image to the expected shape
    smallimg = cv2.resize(image, (160, 80))  # Corrected resizing dimensions
    smallimg = np.array(smallimg)
    smallimg = smallimg[None,:,:,:]

    # Assuming 'model' is defined and loaded somewhere in your code
    prediction = model.predict(smallimg)[0] * 255
    print(prediction)
    lanes.recent_fit.append(prediction)

    if len(lanes.recent_fit) > 5:
        lanes.recent_fit = lanes.recent_fit[1:]

    lanes.avg_fit = np.mean(np.array([i for i in lanes.recent_fit]), axis=0)
    blanks = np.zeros_like(lanes.avg_fit).astype(np.uint8)
    lane_drawn = np.dstack((blanks, lanes.avg_fit, blanks))
    print('lines',lanes.avg_fit)
    lane_image = cv2.resize(lane_drawn, (720, 1280))

    # Resize the original image to match the size of lane_image
    resized_image = cv2.resize(image, (720, 1280))

    # Specify the data type of the output array
    result = cv2.addWeighted(resized_image.astype(np.float32), 1, lane_image.astype(np.float32), 1, 0, dtype=cv2.CV_32F)
    return result



In [ ]:
import cv2
import numpy as np

class Lanes1():
    def __init__(self):
        self.recent_fit =[]
        self.avg_fit =[]

def road_lines(image):
    # Resize the image to the expected shape
    smallimg = cv2.resize(image, (160, 80))  # Corrected resizing dimensions
    smallimg = np.array(smallimg)
    smallimg = smallimg[None,:,:,:]

    # Assuming 'model' is defined and loaded somewhere in your code
    prediction = model.predict(smallimg)[0] * 255
    print("Prediction Values:")
    print(prediction)  # Print the prediction values
    lanes.recent_fit.append(prediction)

    if len(lanes.recent_fit) > 5:
        lanes.recent_fit = lanes.recent_fit[1:]

    lanes.avg_fit = np.mean(np.array([i for i in lanes.recent_fit]), axis=0)
    blanks = np.zeros_like(lanes.avg_fit).astype(np.uint8)
    lane_drawn = np.dstack((blanks, lanes.avg_fit, blanks))

    lane_image = cv2.resize(lane_drawn, (720, 1280))

    # Resize the original image to match the size of lane_image
    resized_image = cv2.resize(image, (720, 1280))

    # Specify the data type of the output array
    result = cv2.addWeighted(resized_image.astype(np.float32), 1, lane_image.astype(np.float32), 1, 0, dtype=cv2.CV_32F)
    return result


In [ ]:
uploaded = files.upload()

In [ ]:
vid_in=VideoFileClip("DetectingLanes.mp4")
vid_out="DetectingLanes1.mp4"
lanes=lanes()
video=vid_in.fl_image(road_lines)
video.write_videofile(vid_out)